# Unified Lab 4: MDPs & Dynamic Programming

## The Scenario
**You are an AI Architect at "AeroLogistics."** Before the company's autonomous drones can learn on the fly, you are designing the pathfinding logic for the ground-based warehouse rovers. You possess a complete blueprint of a specific warehouse zone (a 4x4 grid). Your task is to implement an algorithm that calculates the optimal path to a charging station while avoiding a dangerous trap. You must ensure your navigation system remains robust even if the rovers suffer from mechanical slip (uncertainty) on the slick warehouse floor.



[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/unified-labs/blob/main/the_dynamic_programmer/Module4-Lab.ipynb)

## Milestone 1 (The Baseline): The Deterministic MDP & Value Iteration
**Objective:** Define the warehouse as an MDP and use Value Iteration to find the optimal action-value function when the environment is perfectly predictable.

### The Architect's Blueprint
Instruct your AI assistant to build a Python class representing a 4x4 Gridworld MDP with the following constraints:
* **States & Actions:** A NxN grid with actions {Up, Down, Left, Right}. Assume the agent cannot walk through walls; invalid moves leave it in its current state.
* **The Map:** Start at the bottom-left `(0,0)`. The Goal is at the bottom-right `(0,3)` with a Reward of `+10`. There are Traps at `(0,1)` and `(0,2)` with a Reward of `-100`. Every other step costs `-1`.
* **Transitions:** The environment must be strictly deterministic (i.e., $P(s'|s,a) = 1$).
* **The Solver:** Instruct the AI to write a `value_iteration` function using a discount factor of $\gamma = 0.9$. It must implement the Bellman Optimality Equation: $V^*(s) = \max_a \sum_{s'} P(s'|s,a)[R(s,a,s') + \gamma V^*(s')]$.
* Output the final converged Value matrix and the resulting optimal Policy (the best action for each state).

In [1]:
# ============================================================
# MILESTONE 1: Deterministic 4x4 Gridworld MDP + Value Iteration
# ============================================================
# Convention: a state is (row, col) with row 0 = BOTTOM row, so 'Up' increases
# the row index. This matches the harness: from (0,0), 'Up' must lead to (1,0).
#
#   row 3 | (3,0)  (3,1)  (3,2)  (3,3)
#   row 2 | (2,0)  (2,1)  (2,2)  (2,3)
#   row 1 | (1,0)  (1,1)  (1,2)  (1,3)
#   row 0 | START  TRAP   TRAP   GOAL

ACTIONS = ('Up', 'Down', 'Left', 'Right')
MOVES = {'Up': (1, 0), 'Down': (-1, 0), 'Left': (0, -1), 'Right': (0, 1)}


class GridworldMDP:
    """A deterministic NxN Gridworld MDP: (S, A, P, R, gamma)."""

    def __init__(self, size=4, start=(0, 0), goal=(0, 3), traps=((0, 1), (0, 2)),
                 goal_reward=10.0, trap_reward=-100.0, step_reward=-1.0):
        self.size = size
        self.states = [(r, c) for r in range(size) for c in range(size)]   # S
        self.actions = list(ACTIONS)                                       # A
        self.start = start
        self.goal = goal
        self.traps = set(traps)
        self.goal_reward = goal_reward
        self.trap_reward = trap_reward
        self.step_reward = step_reward

    def is_terminal(self, state):
        """Goal and traps absorb the rover: the episode ends there."""
        return state == self.goal or state in self.traps

    def _move(self, state, action):
        """Where `action` points from `state`. Walls bounce the rover back."""
        r, c = state
        dr, dc = MOVES[action]
        nr, nc = r + dr, c + dc
        if 0 <= nr < self.size and 0 <= nc < self.size:
            return (nr, nc)
        return state          # invalid move -> stay put

    def get_transition_prob(self, state, action):
        """P(s'|s,a) as a dict {next_state: probability}. Deterministic: one outcome."""
        if self.is_terminal(state):
            return {state: 1.0}
        return {self._move(state, action): 1.0}

    def get_reward(self, state, action, next_state):
        """R(s,a,s') -- the reward is earned by ENTERING next_state."""
        if self.is_terminal(state):
            return 0.0
        if next_state == self.goal:
            return self.goal_reward
        if next_state in self.traps:
            return self.trap_reward
        return self.step_reward


def q_value(env, V, state, action, gamma):
    """One-step lookahead: sum_{s'} P(s'|s,a) [ R(s,a,s') + gamma * V(s') ]."""
    return sum(p * (env.get_reward(state, action, nxt) + gamma * V[nxt])
               for nxt, p in env.get_transition_prob(state, action).items())


def value_iteration(env, gamma=0.9, theta=1e-6):
    """Bellman optimality: V*(s) = max_a sum_s' P(s'|s,a)[R(s,a,s') + gamma V*(s')].

    Returns (V, policy, sweeps). Terminal states keep V = 0 and policy None.
    """
    V = {s: 0.0 for s in env.states}
    sweeps = 0
    while True:
        delta = 0.0
        sweeps += 1
        for s in env.states:
            if env.is_terminal(s):
                continue
            v_old = V[s]
            V[s] = max(q_value(env, V, s, a, gamma) for a in env.actions)
            delta = max(delta, abs(v_old - V[s]))
        if delta < theta:                 # values have converged
            break

    # Read the greedy policy off the converged values.
    policy = {s: (None if env.is_terminal(s)
                  else max(env.actions, key=lambda a: q_value(env, V, s, a, gamma)))
              for s in env.states}
    return V, policy, sweeps


# ---------- display helpers (row 3 printed first so it reads like the warehouse) ----------
ARROWS = {'Up': 'Up', 'Down': 'Down', 'Left': 'Left', 'Right': 'Right', None: '--'}


def show_values(env, V, title):
    print(f"{title}\n" + "-" * len(title))
    for r in range(env.size - 1, -1, -1):
        print('  '.join(f"{V[(r, c)]:8.2f}" for c in range(env.size)))
    print()


def show_policy(env, policy, title):
    print(f"{title}\n" + "-" * len(title))
    for r in range(env.size - 1, -1, -1):
        cells = []
        for c in range(env.size):
            s = (r, c)
            if s == env.goal:
                cells.append('GOAL')
            elif s in env.traps:
                cells.append('TRAP')
            else:
                cells.append(ARROWS[policy[s]])
        print('  '.join(f"{x:>6}" for x in cells))
    print()


def trace_path(env, policy, max_steps=25):
    """Follow the greedy policy from the start, ignoring slip (intended moves only)."""
    s, path = env.start, [env.start]
    for _ in range(max_steps):
        if env.is_terminal(s):
            break
        s = env._move(s, policy[s])
        path.append(s)
    return path


# ---------- solve Milestone 1 ----------
GAMMA = 0.9
env = GridworldMDP()
V_det, policy_det, sweeps_det = value_iteration(env, gamma=GAMMA)

show_values(env, V_det, "MILESTONE 1 - Converged V* (deterministic)")
show_policy(env, policy_det, "MILESTONE 1 - Optimal policy")
print(f"Converged in {sweeps_det} sweeps.")
print(f"V*(start {env.start}) = {V_det[env.start]:.4f}")
print("Greedy path:", ' -> '.join(str(s) for s in trace_path(env, policy_det)))

MILESTONE 1 - Converged V* (deterministic)
------------------------------------------
    1.81      3.12      4.58      6.20
    3.12      4.58      6.20      8.00
    4.58      6.20      8.00     10.00
    3.12      0.00      0.00      0.00

MILESTONE 1 - Optimal policy
----------------------------
  Down    Down    Down    Down
  Down    Down    Down    Down
 Right   Right   Right    Down
    Up    TRAP    TRAP    GOAL

Converged in 6 sweeps.
V*(start (0, 0)) = 3.1220
Greedy path: (0, 0) -> (1, 0) -> (1, 1) -> (1, 2) -> (1, 3) -> (0, 3)


### The Test Harness:
*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [2]:
# MILESTONE 1 TEST HARNESS - DO NOT MODIFY

# Test 1: Verify MDP components
assert hasattr(env, 'states'), "Environment must define a set of states S."
assert hasattr(env, 'actions'), "Environment must define a set of actions A."
assert callable(env.get_transition_prob), "Must define a transition function P(s'|s,a)."
assert callable(env.get_reward), "Must define a reward function R(s,a,s')."

# Test 2: Verify Deterministic Transitions
# Assuming state (0,0) is bottom-left, action 'Up' should have 1.0 prob of moving to (1,0)
probs = env.get_transition_prob(state=(0,0), action='Up')
assert probs.get((1,0), 0) == 1.0, "Milestone 1 transitions must be 100% deterministic."

### Architect's Audit (Milestone 1)

1. Run your generated solver and examine the output. Look at the specific path the policy dictates from the Start state `(0,0)` to the Goal `(0,3)`. Does the rover take a route that skirts directly adjacent to the Trap, or does it take a wider path along the outside edge of the grid? Given the mathematical rules of this specific deterministic MDP, why is the solver choosing this exact route?

### My Answer

**The rover skirts directly adjacent to the traps.** The optimal policy sends it

`(0,0) -> (1,0) -> (1,1) -> (1,2) -> (1,3) -> (0,3)`

which runs along row 1 — the cells sitting *directly on top of* both trap cells `(0,1)` and `(0,2)` — rather than detouring up to row 2 or 3. It hugs the danger as closely as the grid allows.

**Why the solver chooses this route.** In a deterministic MDP, $P(s'|s,a) = 1$ for exactly one successor, so *being next to a trap has no cost at all*. The $-100$ can only enter a Q-value if the agent deliberately selects the action that steps into the trap cell, and the $\max_a$ operator discards that branch immediately. Proximity is not risk when there is no chance of an unintended move — the trap penalty is simply never sampled along the chosen path.

That leaves the $-1$ step cost and $\gamma = 0.9$ as the only forces acting on the policy, and both reward **brevity**. So the problem collapses to "find the shortest legal route," and 5 moves is the minimum:

$$V^*(0,0) = -1 - 0.9 - 0.81 - 0.729 + 0.9^4(10) = -3.439 + 6.561 = 3.122$$

which is exactly the converged value the solver printed for the start state. A wider path along the outside edge costs at least 2 extra moves, which both adds discounted step penalties and pushes the $+10$ two more factors of $\gamma$ into the future ($0.9^6 \cdot 10 = 5.31$ instead of $0.9^4 \cdot 10 = 6.56$). Strictly worse. With certainty in the transition model, distance is the only currency the Bellman equation trades in.

## Milestone 2 (The Stress Test): The Stochastic Reality
**Objective:** Introduce uncertainty into the transition matrix and observe how the optimal policy adapts to risk.

### The Architect's Blueprint

The warehouse floor is slick, meaning the rover's motors are no longer perfectly reliable. Instruct your AI to create a new environment class (`StochasticGridworldMDP`) that inherits from or duplicates your Milestone 1 environment, but alters the transition function to be stochastic:
* When the agent chooses an action, it has an **80% chance** of moving in the intended direction.
* The agent has a **20% chance** of slipping down. That is, no matter which action the agent takes ('Up', 'Right', or 'Left'), it always has a 20% chance of actually moving 'Down'.
* Invalid moves (hitting a wall) still leave the agent in its current state.
* Instantiate this new environment as `env_stochastic`, run the exact same `value_iteration` function from Milestone 1 on it, and print the new optimal policy.

In [3]:
# ============================================================
# MILESTONE 2: The slick floor -- stochastic transitions
# ============================================================
# Same map, same rewards, same solver. Only P(s'|s,a) changes:
#   80% the rover moves where it was told to go,
#   20% the motors slip and it moves DOWN instead, whatever it intended.
# Walls still bounce it back to its current cell.

class StochasticGridworldMDP(GridworldMDP):
    """Milestone 1's warehouse, but the floor is slick."""

    def __init__(self, *args, p_intended=0.8, p_slip=0.2, **kwargs):
        super().__init__(*args, **kwargs)
        self.p_intended = p_intended
        self.p_slip = p_slip

    def get_transition_prob(self, state, action):
        """P(s'|s,a): 0.8 on the intended move, 0.2 on a downward slip.

        Outcomes are accumulated, not overwritten -- if the intended move and the
        slip land on the same cell (e.g. action 'Down', or both blocked by the
        same wall), that cell correctly carries the combined probability.
        """
        if self.is_terminal(state):
            return {state: 1.0}
        probs = {}
        for nxt, p in ((self._move(state, action), self.p_intended),
                       (self._move(state, 'Down'), self.p_slip)):
            probs[nxt] = probs.get(nxt, 0.0) + p
        return probs


# ---------- solve Milestone 2 with the SAME value_iteration function ----------
env_stochastic = StochasticGridworldMDP()
V_sto, policy_sto, sweeps_sto = value_iteration(env_stochastic, gamma=GAMMA)

show_values(env_stochastic, V_sto, "MILESTONE 2 - Converged V* (stochastic, 20% slip)")
show_policy(env_stochastic, policy_sto, "MILESTONE 2 - Optimal policy")
print(f"Converged in {sweeps_sto} sweeps (deterministic case took {sweeps_det}).")
print(f"V*(start) deterministic = {V_det[env.start]:.4f}")
print(f"V*(start) stochastic    = {V_sto[env_stochastic.start]:.4f}")
print(f"Drop in start-state value = {V_det[env.start] - V_sto[env_stochastic.start]:.4f}\n")
print("Intended-move path, deterministic:",
      ' -> '.join(str(s) for s in trace_path(env, policy_det)))
print("Intended-move path, stochastic:   ",
      ' -> '.join(str(s) for s in trace_path(env_stochastic, policy_sto)))
print("\nExample distributions:")
print("  P(.|(0,0),'Up') =", env_stochastic.get_transition_prob(state=(0, 0), action='Up'))
print("  P(.|(1,1),'Right') =", env_stochastic.get_transition_prob(state=(1, 1), action='Right'),
      " <- 20% straight into the trap at (0,1)")

MILESTONE 2 - Converged V* (stochastic, 20% slip)
-------------------------------------------------
   -0.51      1.17      3.88      6.20
   -1.92     -3.48      2.31      8.00
   -3.09    -23.02    -13.60     10.00
   -3.93      0.00      0.00      0.00

MILESTONE 2 - Optimal policy
----------------------------
 Right   Right   Right    Down
    Up   Right   Right    Down
    Up    Left   Right    Down
    Up    TRAP    TRAP    GOAL

Converged in 24 sweeps (deterministic case took 6).
V*(start) deterministic = 3.1220
V*(start) stochastic    = -3.9329
Drop in start-state value = 7.0549

Intended-move path, deterministic: (0, 0) -> (1, 0) -> (1, 1) -> (1, 2) -> (1, 3) -> (0, 3)
Intended-move path, stochastic:    (0, 0) -> (1, 0) -> (2, 0) -> (3, 0) -> (3, 1) -> (3, 2) -> (3, 3) -> (2, 3) -> (1, 3) -> (0, 3)

Example distributions:
  P(.|(0,0),'Up') = {(1, 0): 0.8, (0, 0): 0.2}
  P(.|(1,1),'Right') = {(1, 2): 0.8, (0, 1): 0.2}  <- 20% straight into the trap at (0,1)


### The Test Harness

*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [4]:
# MILESTONE 2 TEST HARNESS - DO NOT MODIFY

# Test 1: Verify Stochastic Transitions
probs = env_stochastic.get_transition_prob(state=(0,0), action='Up')
assert probs.get((1,0), 0) == 0.8, "Intended direction must have an 80% probability."
assert len(probs) > 1, "Action must result in a probability distribution over multiple states."
assert round(sum(probs.values()), 5) == 1.0, "Probabilities must sum to 1.0."

### The Architect's Audit (Milestone 2)

1. Compare the new policy printed for `env_stochastic` to the policy generated in Milestone 1. Specifically, look at the route the rover now takes from the Start state `(0,0)`. Does it still walk directly adjacent to the Trap? Mathematically, how does the inclusion of the 20% slip probabilities in the Bellman equation force this behavioral change, even though the trap's penalty (-100) remained exactly the same?

### My Answer

**No — the rover abandons row 1 entirely and climbs to the top edge.** The new policy is

`(0,0) -> (1,0) -> (2,0) -> (3,0) -> (3,1) -> (3,2) -> (3,3) -> (2,3) -> (1,3) -> (0,3)`

That is 9 intended moves instead of 5.

**The mathematics that forces the change.** The penalty is still exactly $-100$; what changed is that it is now reachable *with non-zero probability from cells the agent only passes near*. The Bellman backup is no longer a single successor but an expectation over the distribution:

$$V^*(s) = \max_a \sum_{s'} P(s'|s,a)\,[R(s,a,s') + \gamma V^*(s')]$$

From any cell in row 1, **every** action — `Up`, `Left`, `Right` alike — carries $P(\text{trap}\,|\,s,a) = 0.2$, because the slip is unconditional. So every Q-value in that row pays

$$0.2 \times (-100) = -20$$

in expectation before any other term is counted. Expected reward is the *product* $P \times R$: holding $R$ fixed and lifting $P$ from $0$ to $0.2$ manufactures a $-20$ term out of nothing. That is why the converged values show $V^*(1,1) = -23.02$ and $V^*(1,2) = -13.60$ — the row that was optimal in Milestone 1 is now the worst real estate in the warehouse.

The policy did not become cautious because anyone told it to respect the trap. Risk-aversion **fell out of the arithmetic** the moment the transition function stopped being certain.

## Milestone 3 (The Insight): Policy Iteration
**Objective:** Replace Value Iteration with Policy Iteration, explicitly separating the evaluation of a policy from its improvement, and analyze the computational differences.

### The Architect's Blueprint
Value Iteration merges evaluation and improvement into one continuous sweep. Instruct your AI to build a new solver class (`PolicyIterationSolver`) that formally separates these concepts.
* It must have a discrete `policy_evaluation` method that computes $V^\pi(s)$ under the *current* policy until convergence (using a threshold `theta=1e-6`).
* It must have a discrete `policy_improvement` method that updates the policy to greedily choose better actions based on the newly calculated values.
* The `solve` method should alternate between these two steps until the policy stops changing.
* Have the AI instrument the `solve` method to print out the total number of *outer loop* iterations (policy improvements) it took to converge.
* Solve the stochastic environment from Milestone 2 using this new method and print the final policy.

In [5]:
# ============================================================
# MILESTONE 3: Policy Iteration -- evaluation and improvement, kept separate
# ============================================================

class PolicyIterationSolver:
    """Alternates policy_evaluation and policy_improvement until the policy is stable."""

    def __init__(self, gamma=0.9, theta=1e-6, verbose=True):
        self.gamma = gamma
        self.theta = theta
        self.verbose = verbose
        self.n_outer = 0          # outer loop iterations (policy improvements)
        self.n_eval_sweeps = 0    # inner sweeps summed over all evaluations

    def policy_evaluation(self, env, policy, V=None):
        """Compute V^pi by sweeping the Bellman EXPECTATION equation to convergence.

        No max over actions here -- the action is dictated by the current policy:
            V^pi(s) = sum_s' P(s'|s,pi(s)) [ R(s,pi(s),s') + gamma V^pi(s') ]
        """
        V = dict(V) if V is not None else {s: 0.0 for s in env.states}
        while True:
            delta = 0.0
            self.n_eval_sweeps += 1
            for s in env.states:
                if env.is_terminal(s):
                    continue
                v_old = V[s]
                V[s] = q_value(env, V, s, policy[s], self.gamma)
                delta = max(delta, abs(v_old - V[s]))
            if delta < self.theta:
                return V

    def policy_improvement(self, env, V, policy):
        """Act greedily w.r.t. V. Returns (new_policy, stable) -- stable if nothing changed."""
        stable = True
        new_policy = {}
        for s in env.states:
            if env.is_terminal(s):
                new_policy[s] = None
                continue
            new_policy[s] = max(env.actions, key=lambda a: q_value(env, V, s, a, self.gamma))
            if new_policy[s] != policy[s]:
                stable = False
        return new_policy, stable

    def solve(self, env):
        """Alternate evaluation and improvement until the policy stops changing."""
        self.n_outer = 0
        self.n_eval_sweeps = 0
        # Naive starting policy: always 'Up'.
        policy = {s: (None if env.is_terminal(s) else env.actions[0]) for s in env.states}
        V = {s: 0.0 for s in env.states}
        while True:
            V = self.policy_evaluation(env, policy, V)
            self.n_outer += 1
            policy, stable = self.policy_improvement(env, V, policy)
            if stable:
                break
        if self.verbose:
            print(f"Policy Iteration converged in {self.n_outer} OUTER loop iterations "
                  f"(policy improvements),")
            print(f"  using {self.n_eval_sweeps} total policy-evaluation sweeps.")
        return V, policy


# ---------- solve the Milestone 2 environment with Policy Iteration ----------
solver = PolicyIterationSolver(gamma=GAMMA, theta=1e-6)
V_pi, policy_pi = solver.solve(env_stochastic)

show_values(env_stochastic, V_pi, "MILESTONE 3 - V^pi* from Policy Iteration")
show_policy(env_stochastic, policy_pi, "MILESTONE 3 - Optimal policy (Policy Iteration)")

# ---------- compare against Value Iteration from Milestone 2 ----------
print("Policy matches Value Iteration exactly:", policy_pi == policy_sto)
print(f"Largest value disagreement: {max(abs(V_pi[s] - V_sto[s]) for s in env_stochastic.states):.2e}\n")
print(f"Value Iteration : {sweeps_sto} sweeps (each sweep maximises over all 4 actions)")
print(f"Policy Iteration: {solver.n_outer} outer iterations, "
      f"but {solver.n_eval_sweeps} evaluation sweeps underneath")

Policy Iteration converged in 7 OUTER loop iterations (policy improvements),
  using 171 total policy-evaluation sweeps.
MILESTONE 3 - V^pi* from Policy Iteration
-----------------------------------------
   -0.51      1.17      3.88      6.20
   -1.92     -3.48      2.31      8.00
   -3.09    -23.02    -13.60     10.00
   -3.93      0.00      0.00      0.00

MILESTONE 3 - Optimal policy (Policy Iteration)
-----------------------------------------------
 Right   Right   Right    Down
    Up   Right   Right    Down
    Up    Left   Right    Down
    Up    TRAP    TRAP    GOAL

Policy matches Value Iteration exactly: True
Largest value disagreement: 2.73e-07

Value Iteration : 24 sweeps (each sweep maximises over all 4 actions)
Policy Iteration: 7 outer iterations, but 171 evaluation sweeps underneath


### The Test Harness
*Use this exact block of code; your AI's generated code MUST pass these assertions.*

In [6]:
# MILESTONE 3 TEST HARNESS - DO NOT MODIFY

assert hasattr(solver, 'policy_evaluation'), "Solver must explicitly contain a policy evaluation step."
assert hasattr(solver, 'policy_improvement'), "Solver must explicitly contain a policy improvement step."

# Test the solver on the stochastic environment from Milestone 2
V_pi, policy_pi = solver.solve(env_stochastic)
assert len(policy_pi) == 16, "Policy must cover all 16 states of the 4x4 grid."

Policy Iteration converged in 7 OUTER loop iterations (policy improvements),
  using 171 total policy-evaluation sweeps.


### Architect's Audit (Milestone 3)

1. Look at the final policy generated by your Policy Iteration solver. Does it perfectly match the policy generated by Value Iteration in Milestone 2? Now, look at the number of outer loop iterations it took to converge. How does this number compare to the total number of sweeps Value Iteration typically requires?

### My Answer

**Yes, the policies match exactly.** `policy_pi == policy_sto` returns `True` for all 16 states, and the two value functions agree to within $2.73 \times 10^{-7}$ — just the residual slack of the `theta=1e-6` convergence threshold, not a real disagreement. This is the expected result: both algorithms are searching for the fixed point of the *same* Bellman optimality equation, and for $\gamma < 1$ that fixed point is unique. They differ only in the route they take to it, never in the destination.

**The iteration counts, and why the comparison is a trap.** Policy Iteration converged in **7 outer loop iterations** against Value Iteration's **24 sweeps** — apparently a 3.4× win. But the instrumentation shows that headline is measuring two different units of work:

| | Outer iterations | Total sweeps over the state space | Actual Bellman backups |
|---|---|---|---|
| Value Iteration | 24 | **24** | **1300** |
| Policy Iteration | 7 | **171** | **2587** |

What the low outer count genuinely reflects is that each policy improvement is made against an *exactly converged* $V^\pi$, so the greedy step extracts the maximum possible information and the policy locks in after very few updates. Value Iteration improves its implicit policy a little on every sweep, using values that are still half-propagated. Fewer, better-informed decisions versus many cheap ones — not "faster."

## The Summary Audit

To complete the lab, submit your generated notebook along with brief, 3-4 sentence answers to the following questions:

1. **Data-Binding (The Value of Uncertainty):** Look at your printed Value matrices for Milestone 1 (Deterministic) and Milestone 2 (Stochastic). Identify the numerical value of the Start state `(0,0)` in both matrices. Explain *mathematically*, referencing the Bellman equation and your specific transition probabilities, why the value of the Start state dropped when the downward slip was introduced, even though the trap penalty itself did not change.
2. **Adversarial Critique:** Feed the following prompt into your preferred LLM: *"In Reinforcement Learning, is Policy Iteration generally faster and computationally cheaper per iteration than Value Iteration?"* The LLM will likely give a nuanced but potentially misleading answer. Based on Lecture 4, critique the LLM's response. Specifically, identify why the `policy_evaluation` step makes Policy Iteration computationally heavy *per sweep* compared to Value Iteration.

### My Answers

#### 1. Data-Binding — The Value of Uncertainty

The state evolves from **$V^*(0,0) = 3.1220$** to **$V^*(0,0) = -3.9329$** — a decrease of **7.0549**, and a change in sign. Expanding state under the optimal action `Up` shows how it evolves:

$$V^*(0,0) = 0.8\,[-1 + 0.9\,V^*(1,0)] + 0.2\,[-1 + 0.9\,V^*(0,0)] = 0.8(-3.781) + 0.2(-4.537) = -3.93$$

In one in five times the rover slides to the south wall, pays the $-1$ penalty, and ends up back where it began. The value it's backing up from itself collapes: $V^*(1,0)$ drops from $4.58$ to $-3.09$ (its neighbour $V^*(1,1)$ goes from $6.20$ to $-23.02$). The $\gamma V^*(s')$ term drags everything back to the initial state. Since the policy takes a longer route (with 9 moves instead of 5) the $+10$ is discounted by $\gamma^9 \approx 0.387$ rather than $\gamma^4 \approx 0.656$. In short, the expected reward is the **product** $P \times R$, and Bellman maximises over *expectations*, not best cases.

#### 2. Adversarial Critique — "Is Policy Iteration faster and cheaper per iteration than Value Iteration?"

Policy Iteration's iteration is an *outer* loop: one complete policy-evaluation run to convergence followed by an improvement pass. In contrast, a Value Iteration iteration is a *single sweep*, so the two are not directly comparable. A single evaluation sweep may indeed be cheaper than a single VI sweep: with the action fixed by $\pi$, policy evaluation applies the Bellman *expectation* equation and performs one backup per state, whereas Value Iteration's $\max_a$ requires $|A| = 4$ backups per state. That is roughly a 4× saving **per sweep**. However, "per iteration" means something different for Policy Iteration. Before the policy can be improved even once, `policy_evaluation` must repeatedly sweep the entire state space until $\Delta < \theta = 10^{-6}$. Thus, one outer iteration bundles together a whole stack of these inexpensive sweeps. The bundle—not an individual sweep—is what counts as an iteration. The lecture's claim that policy iteration "reuses structure" explains why its *outer* iteration count is low; it does not claim that the cost per unit of work is lower, as the LLM suggests.

Policy Iteration converges in fewer *policy updates*, and each update is more greedy (hence better informed) with respect to a fully converged $V^\pi$. It is preferable when a good initial policy is available, when $|A|$ is large enough for the $max$ operation to dominate, or when evaluation is solved directly as a linear system (though that approach is $O(|S|^3)$ and is even less practical for large state spaces). Otherwise, the practical compromise is **modified (truncated) policy iteration**: limit evaluation to a few sweeps rather than running it to $\theta$. From this perspective, Value Iteration is the limiting case of Policy Iteration in which evaluation is truncated to a single sweepwhich is why there is no answer to "which is faster" independent of $|A|$, $\gamma$, and $\theta$.